# TP4 — Réseaux de neurones multicouches
**Cours IA & Cybersécurité — Master 1**

---

## Ce que vous allez apprendre

Dans le TP1, vous avez construit un **perceptron** : un seul neurone capable de séparer deux classes *si elles sont séparables par une ligne droite*.

Mais que se passe-t-il quand le problème est plus complexe ?  
→ On **empile plusieurs couches de neurones**. C'est ce qu'on appelle un **réseau de neurones multicouches** (en anglais : *Multi-Layer Perceptron*, ou **MLP**).

À la fin de ce TP, vous saurez :
1. Pourquoi un seul perceptron ne suffit pas toujours
2. Comment fonctionne un MLP (couches, poids, activation)
3. Ce qu'est la **rétropropagation** (comment le réseau apprend de ses erreurs)
4. Implémenter un MLP avec **PyTorch** pour classer du trafic réseau

**Durée estimée** : 2h  
**Prérequis** : TP1 (perceptron) complété

---

## Partie 1 — La limite du perceptron

### 1.1 Rappel : ce que fait un perceptron

Un perceptron :
- prend des entrées numériques (ex : durée d'une connexion, nombre de paquets)
- les multiplie par des **poids**
- fait la somme
- applique une **fonction d'activation** (ex : 1 si positif, 0 sinon)

Il trace une **frontière droite** entre les classes. Ça fonctionne bien si les classes sont séparables linéairement.

### 1.2 Un problème que le perceptron ne peut PAS résoudre

Imaginez ce scénario de sécurité réseau :

> Une connexion est **suspecte** si elle a beaucoup de paquets ET peu de données — OU peu de paquets ET beaucoup de données.  
> Mais elle est **normale** si les deux sont élevés ou les deux sont faibles.

Ce motif ressemble au problème mathématique du **XOR** (ou exclusif) :  
- (0,0) → normal
- (0,1) → suspect
- (1,0) → suspect
- (1,1) → normal

Aucune droite ne peut séparer ces deux classes. **Un seul perceptron échoue.**

In [ ]:
# Visualisons pourquoi le XOR est impossible à séparer linéairement
import numpy as np
import matplotlib.pyplot as plt

# Les 4 points XOR
points  = np.array([[0,0], [0,1], [1,0], [1,1]])
labels  = np.array([0, 1, 1, 0])  # 0 = normal, 1 = suspect
couleurs = ['green' if l == 0 else 'red' for l in labels]

plt.figure(figsize=(5, 4))
plt.scatter(points[:,0], points[:,1], c=couleurs, s=200, zorder=5)
plt.title("Problème XOR — aucune droite ne sépare vert et rouge")
plt.xlabel("Beaucoup de paquets ?")
plt.ylabel("Beaucoup de données ?")
plt.xticks([0,1], ['Non','Oui'])
plt.yticks([0,1], ['Non','Oui'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Vert = normal, Rouge = suspect")
print("Essayez mentalement de tracer une droite... impossible !")

### 1.3 La solution : ajouter une couche cachée

L'idée est simple : au lieu d'un seul neurone, on **empile des couches** :

```
Entrées  →  [Couche cachée]  →  [Couche de sortie]  →  Prédiction
```

La couche cachée "transforme" l'espace des données pour que la sortie puisse ensuite tracer une frontière — même courbe, même complexe.

> **Analogie aéronautique** : un pilote automatique basique (perceptron) gère les vols par beau temps. Pour les conditions complexes (turbulences, pannes multiples), il faut un système multicouche où chaque couche gère un niveau d'abstraction différent.

---
## Partie 2 — Architecture d'un MLP

### 2.1 Les trois types de couches

| Couche | Rôle | Exemple |
|--------|------|---------|
| **Entrée** | Reçoit les données brutes | durée, nb_paquets, taille_données... |
| **Cachée(s)** | Transforme et détecte des motifs | combinaisons de caractéristiques |
| **Sortie** | Donne la prédiction finale | 0 = normal, 1 = attaque |

### 2.2 Les fonctions d'activation

Dans le perceptron, l'activation était binaire (0 ou 1). Dans un MLP, on utilise des fonctions plus douces :

- **ReLU** (*Rectified Linear Unit*) : `f(x) = max(0, x)` — la plus courante dans les couches cachées
- **Sigmoid** : `f(x) = 1 / (1 + e^-x)` — donne une probabilité entre 0 et 1 (sortie)

> **Pourquoi ReLU ?** Elle est simple, rapide, et évite un problème mathématique appelé *vanishing gradient* (les gradients qui disparaissent dans les couches profondes).

In [ ]:
# Visualisons ReLU et Sigmoid
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-4, 4, 200)

relu    = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.plot(x, relu, color='steelblue', linewidth=2)
ax1.set_title("ReLU — couches cachées")
ax1.set_xlabel("x")
ax1.axhline(0, color='gray', linewidth=0.5)
ax1.axvline(0, color='gray', linewidth=0.5)
ax1.grid(alpha=0.3)

ax2.plot(x, sigmoid, color='coral', linewidth=2)
ax2.set_title("Sigmoid — couche de sortie")
ax2.set_xlabel("x")
ax2.axhline(0.5, color='gray', linewidth=0.5, linestyle='--')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("ReLU : tout ce qui est négatif devient 0 → le neurone 'ne s'active pas'")
print("Sigmoid : compresse la sortie entre 0 et 1 → interprétable comme une probabilité")

---
## Partie 3 — Comment le réseau apprend : la rétropropagation

### 3.1 L'intuition

> **Analogie** : imaginez un avion qui rate sa piste à l'atterrissage. Le pilote corrige la trajectoire. Mais comment ? Il regarde *quelle décision* a causé l'erreur (descente trop rapide ? angle trop fort ?) et ajuste chaque paramètre en conséquence.

La **rétropropagation** (*backpropagation*) fait exactement ça pour un réseau de neurones :

1. Le réseau fait une prédiction (passe **avant**, *forward pass*)
2. On mesure l'erreur (la **loss** / perte)
3. On calcule *quelle couche a contribué à quelle erreur* (passe **arrière**, *backward pass*)
4. On ajuste les poids en conséquence (descente de gradient)

### 3.2 La loss (fonction de perte)

Pour un problème de classification binaire (normal / attaque), on utilise la **Binary Cross-Entropy** :

```
Loss = - [ y × log(ŷ) + (1-y) × log(1-ŷ) ]
```

- `y` = la vraie réponse (0 ou 1)
- `ŷ` = la prédiction du réseau (entre 0 et 1)
- Plus la perte est petite, mieux le réseau prédit

**Bonne nouvelle** : PyTorch calcule tout ça automatiquement !

---
## Partie 4 — Cas pratique : détecter du trafic réseau anormal

### 4.1 Le contexte

Un opérateur réseau vous confie un jeu de données de connexions.  
Chaque connexion est décrite par **3 caractéristiques** :
- `duree` : durée de la connexion (en secondes)
- `nb_paquets` : nombre de paquets envoyés
- `taille_moy` : taille moyenne des paquets (en octets)

L'objectif : prédire si la connexion est **normale (0)** ou **anormale/attaque (1)**.

### 4.2 Génération du jeu de données simulé

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

# Reproductibilité
np.random.seed(42)
torch.manual_seed(42)

# --- Génération des données ---
N = 500  # nombre de connexions

# Connexions normales : durée longue, paquets moyens, taille normale
normales = np.column_stack([
    np.random.normal(30, 10, N//2),   # durée ~30s
    np.random.normal(50, 15, N//2),   # ~50 paquets
    np.random.normal(500, 100, N//2)  # ~500 octets/paquet
])

# Connexions anormales : durée très courte, nombreux paquets, petite taille (scan de ports)
anormales = np.column_stack([
    np.random.normal(2, 1, N//2),     # durée ~2s
    np.random.normal(200, 50, N//2),  # ~200 paquets
    np.random.normal(60, 20, N//2)    # ~60 octets/paquet
])

X = np.vstack([normales, anormales]).astype(np.float32)
y = np.array([0]*(N//2) + [1]*(N//2), dtype=np.float32)

print(f"Jeu de données : {X.shape[0]} connexions, {X.shape[1]} caractéristiques")
print(f"Normales : {(y==0).sum()} | Anormales : {(y==1).sum()}")
print(f"\nExemple connexion normale : durée={X[0,0]:.1f}s, paquets={X[0,1]:.0f}, taille={X[0,2]:.0f} oct")
print(f"Exemple connexion anormale : durée={X[N//2,0]:.1f}s, paquets={X[N//2,1]:.0f}, taille={X[N//2,2]:.0f} oct")

### 4.3 Préparation des données

Avant d'entraîner un réseau, on **normalise** les données.  
Pourquoi ? Les durées sont en secondes (ordre 2–30), les tailles en octets (ordre 60–500).  
Sans normalisation, les poids associés aux tailles "domineraient" l'apprentissage.

> **Analogie** : comparer des vitesses en km/h et des altitudes en mètres sans conversion — l'altitude aurait un poids artificiel juste à cause de l'échelle.

In [ ]:
# Découpage train/test (80% entraînement, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalisation (moyenne = 0, écart-type = 1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)  # IMPORTANT : on utilise le scaler du train !

# Conversion en tenseurs PyTorch
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)  # shape (N,1) pour la loss
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test).unsqueeze(1)

print(f"Taille train : {X_train_t.shape}")
print(f"Taille test  : {X_test_t.shape}")
print(f"\nAprès normalisation, 1ère connexion train : {X_train_t[0].numpy()}")

---
## Partie 5 — Construction du MLP avec PyTorch

### 5.1 Architecture choisie

```
Entrée (3 neurones)
      ↓
Couche cachée 1 : 16 neurones + ReLU
      ↓
Couche cachée 2 : 8 neurones + ReLU
      ↓
Sortie : 1 neurone + Sigmoid
      ↓
  Probabilité [0,1]
```

PyTorch permet de décrire cette architecture avec `nn.Sequential` :  
on liste les couches **dans l'ordre**, comme des wagons d'un train.

In [ ]:
# ============================================================
#  À COMPLÉTER
# ============================================================
# Construisez le réseau en suivant l'architecture décrite.
#
# nn.Linear(a, b) = couche linéaire de 'a' entrées vers 'b' sorties
# nn.ReLU()       = activation ReLU
# nn.Sigmoid()    = activation Sigmoid

model = nn.Sequential(
    nn.Linear(3, 16),   # entrée : 3 caractéristiques → 16 neurones
    nn.ReLU(),
    nn.Linear(???, ???),  # couche cachée 2 : 16 → 8
    nn.ReLU(),
    nn.Linear(???, ???),  # sortie : 8 → 1
    nn.Sigmoid()
)

print("Architecture du réseau :")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nNombre total de paramètres (poids) : {total_params}")

### 5.2 Définir la loss et l'optimiseur

- **Loss** : `BCELoss` (*Binary Cross-Entropy Loss*) — adaptée à la classification binaire
- **Optimiseur** : `Adam` — une version améliorée de la descente de gradient, très couramment utilisée
- **Learning rate** : `0.001` — la taille du pas lors de la mise à jour des poids

In [ ]:
# ============================================================
#  À COMPLÉTER
# ============================================================
# Définissez la fonction de perte et l'optimiseur.
# Utilisez : nn.BCELoss() et torch.optim.Adam()

criterion = ???               # fonction de perte
optimizer = torch.optim.Adam(model.parameters(), lr=???)  # lr = 0.001

print("Loss     :", criterion)
print("Optimiseur:", optimizer)

### 5.3 Boucle d'entraînement

La boucle d'entraînement PyTorch suit **toujours le même schéma en 4 étapes** :

```
Pour chaque époque :
  1. Prédire  → model(X)              (forward pass)
  2. Mesurer  → criterion(pred, y)    (calcul de la loss)
  3. Remonter → loss.backward()       (rétropropagation)
  4. Corriger → optimizer.step()      (mise à jour des poids)
```

> Une **époque** = le réseau a vu l'intégralité des données d'entraînement une fois.

In [ ]:
# ============================================================
#  À COMPLÉTER
# ============================================================
# Complétez les 4 étapes de la boucle d'entraînement.

n_epoques = 100
historique_loss = []

for epoque in range(n_epoques):

    # Étape 1 : prédiction (forward pass)
    predictions = ???(X_train_t)         # passer X_train_t dans le modèle

    # Étape 2 : calcul de la perte
    loss = ???(predictions, y_train_t)   # comparer prédictions et vraies valeurs

    # Étape 3 : remise à zéro des gradients (TOUJOURS faire ça avant backward)
    optimizer.zero_grad()

    # Étape 4 : rétropropagation + mise à jour
    ???.backward()    # calculer les gradients
    ???.step()        # mettre à jour les poids

    historique_loss.append(loss.item())

    if (epoque + 1) % 20 == 0:
        print(f"Époque {epoque+1:3d}/{n_epoques} — Loss : {loss.item():.4f}")

print("\nEntraînement terminé !")

### 5.4 Visualiser la courbe d'apprentissage

La loss doit **diminuer** au fil des époques. C'est le signe que le réseau apprend.  
Si elle ne descend pas (ou remonte), il y a un problème : mauvais learning rate, données mal normalisées, etc.

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(historique_loss, color='steelblue', linewidth=2)
plt.title("Courbe d'apprentissage — Loss au fil des époques")
plt.xlabel("Époque")
plt.ylabel("Loss (Binary Cross-Entropy)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss initiale : {historique_loss[0]:.4f}")
print(f"Loss finale   : {historique_loss[-1]:.4f}")
print(f"Amélioration  : {(1 - historique_loss[-1]/historique_loss[0])*100:.1f}%")

---
## Partie 6 — Évaluation sur le jeu de test

### 6.1 Précision (accuracy)

La précision = proportion de connexions **correctement classifiées**.  
Pour décider de la classe, on applique un **seuil** : si la probabilité > 0.5, c'est une attaque.

In [ ]:
# Passage en mode évaluation (désactive le dropout, batchnorm, etc. si présents)
model.eval()

with torch.no_grad():   # on n'a pas besoin de calculer les gradients en évaluation
    proba_test = model(X_test_t)

# Conversion probabilité → classe (seuil 0.5)
predictions_test = (proba_test >= 0.5).float()

# ============================================================
#  À COMPLÉTER
# ============================================================
# Calculez le nombre de prédictions correctes et la précision.
# Rappel : une prédiction est correcte si predictions_test == y_test_t

nb_corrects = (predictions_test == ???).sum().item()
precision   = ??? / len(y_test_t) * 100

print(f"Connexions de test : {len(y_test_t)}")
print(f"Correctement classifiées : {nb_corrects}")
print(f"Précision : {precision:.1f}%")

### 6.2 Matrice de confusion

La précision seule ne suffit pas. En cybersécurité, deux erreurs n'ont pas le même coût :
- **Faux négatif** (FN) : une attaque classée comme normale → **dangereux !**
- **Faux positif** (FP) : une connexion normale classée comme attaque → gêne, fausse alerte

La **matrice de confusion** montre la répartition des 4 cas :

```
                  Prédit : Normal   Prédit : Attaque
Réel : Normal          TN                FP
Réel : Attaque         FN                TP
```

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred_np = predictions_test.numpy().flatten().astype(int)
y_true_np = y_test_t.numpy().flatten().astype(int)

cm = confusion_matrix(y_true_np, y_pred_np)

# Affichage graphique
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(['Normal (0)', 'Attaque (1)'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Normal (0)', 'Attaque (1)'])
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
ax.set_title('Matrice de confusion')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=16)
plt.tight_layout()
plt.show()

print("\nRapport détaillé :")
print(classification_report(y_true_np, y_pred_np, target_names=['Normal', 'Attaque']))

tn, fp, fn, tp = cm.ravel()
print(f"Vrais Négatifs (TN) : {tn}  — normales bien détectées")
print(f"Faux Positifs   (FP): {fp}  — fausses alarmes")
print(f"Faux Négatifs   (FN): {fn}  — attaques manquées ← critique !")
print(f"Vrais Positifs  (TP): {tp}  — attaques détectées")

---
## Partie 7 — Tester sur une nouvelle connexion

Le vrai test : est-ce que votre modèle peut analyser une connexion inconnue en temps réel ?

In [ ]:
# ============================================================
#  À COMPLÉTER — puis essayez vos propres valeurs !
# ============================================================

def analyser_connexion(duree, nb_paquets, taille_moy):
    """
    Analyse une connexion réseau et retourne si elle est suspecte.
    
    Paramètres :
    - duree      : durée en secondes
    - nb_paquets : nombre de paquets
    - taille_moy : taille moyenne des paquets en octets
    """
    # Préparer les données (normalisation OBLIGATOIRE — même scaler que l'entraînement)
    x_nouveau = np.array([[duree, nb_paquets, taille_moy]], dtype=np.float32)
    x_normalise = scaler.transform(x_nouveau)
    x_tensor = torch.FloatTensor(x_normalise)

    # Prédiction
    model.eval()
    with torch.no_grad():
        probabilite = ???(x_tensor).item()   # passer x_tensor dans le modèle

    # Décision
    classe = "⚠️  ATTAQUE" if probabilite >= 0.5 else "✅ Normale"
    print(f"Connexion : durée={duree}s, paquets={nb_paquets}, taille={taille_moy} oct")
    print(f"Probabilité d'attaque : {probabilite:.1%}")
    print(f"Décision : {classe}")
    print()

# Test 1 : connexion typiquement normale
print("--- Test 1 ---")
analyser_connexion(duree=28, nb_paquets=55, taille_moy=490)

# Test 2 : connexion typiquement anormale (scan de ports)
print("--- Test 2 ---")
analyser_connexion(duree=1, nb_paquets=210, taille_moy=55)

# Test 3 : cas ambigu — à vous de juger !
print("--- Test 3 (ambigu) ---")
analyser_connexion(duree=5, nb_paquets=80, taille_moy=300)

---
## Partie 8 — Pour aller plus loin (optionnel)

### 8.1 Expérimentez !

Modifiez le modèle et observez l'impact sur les résultats :

In [ ]:
# EXPÉRIENCE : que se passe-t-il si on change l'architecture ?
# Essayez :
#   - Plus de neurones dans la couche cachée (ex: 32)
#   - Plus de couches cachées
#   - Un learning rate différent (0.01, 0.0001)
#   - Plus d'époques (200, 500)

# Exemple : modèle plus profond
model_v2 = nn.Sequential(
    nn.Linear(3, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

print("Modèle V2 (plus profond) :")
print(model_v2)
print(f"\nNombre de paramètres : {sum(p.numel() for p in model_v2.parameters())}")
print("\n→ Entraînez ce modèle et comparez sa précision au modèle original !")

---
## Récapitulatif

| Concept | Ce que vous avez fait |
|---------|----------------------|
| **Limite du perceptron** | Visualisé le problème XOR non séparable linéairement |
| **Architecture MLP** | Construit un réseau 3 → 16 → 8 → 1 avec PyTorch |
| **Fonctions d'activation** | Utilisé ReLU (couches cachées) et Sigmoid (sortie) |
| **Entraînement** | Appliqué la boucle forward → loss → backward → step |
| **Évaluation** | Mesuré la précision et analysé la matrice de confusion |
| **Inférence** | Analysé de nouvelles connexions en temps réel |

### Questions de réflexion

1. Dans votre matrice de confusion, quel type d'erreur est le plus dangereux en cybersécurité ? Pourquoi ?
2. Que se passe-t-il si vous entraînez le modèle sans normaliser les données ?
3. Un réseau avec plus de paramètres est-il toujours meilleur ? Qu'est-ce que le **surapprentissage** (*overfitting*) ?
4. Comment ce type de modèle pourrait-il être intégré dans un système de détection d'intrusion réel ?

---
*TP4 — Cours IA & Cybersécurité · M1 · 2025-2026*